<a href="https://colab.research.google.com/github/JaimRM/QuantitativeFinance/blob/main/Tesla_DCF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install openpyxl

In [9]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ─── Color palette (IB standard) ────────────────────────────────────────────
C_BLUE     = "0000FF"     # Hardcoded inputs
C_BLACK    = "000000"     # Formulas
C_GREEN    = "008000"     # Cross-sheet links
C_WHITE    = "FFFFFF"
C_HEADER   = "1F3864"     # Dark navy header
C_SUBHEAD  = "2E5496"     # Medium blue
C_LIGHT    = "D6E4F7"     # Light blue highlight
C_GREY     = "F2F2F2"     # Alternating row
C_YELLOW   = "FFFFD966"   # Key valuation target highlight

# ─── Style helpers ──────────────────────────────────────────────────────────
def font(bold=False, color=C_BLACK, size=10, italic=False):
    return Font(name="Arial", bold=bold, color=color, size=size, italic=italic)

def fill(color):
    return PatternFill("solid", fgColor=color)

def border(left=None, right=None, top=None, bottom=None):
    sides = {k: Side(style=v) for k, v in
             dict(left=left, right=right, top=top, bottom=bottom).items() if v}
    return Border(**sides)

def center():
    return Alignment(horizontal="center", vertical="center", wrap_text=True)

def right():
    return Alignment(horizontal="right", vertical="center")

def left_al():
    return Alignment(horizontal="left", vertical="center", indent=1)

# Number formats
FMT_DOLLAR  = '#,##0;(#,##0);"-"'
FMT_DOLLAR1 = '#,##0.0;(#,##0.0);"-"'
FMT_PCT     = '0.0%;(0.0%);"-"'
FMT_MULT    = '0.0"x";(0.0"x");"-"'

def style_header_cell(ws, cell_ref, value, dark=True):
    c = ws[cell_ref]
    c.value = value
    c.font = font(bold=True, color=C_WHITE, size=10)
    c.fill = fill(C_HEADER if dark else C_SUBHEAD)
    c.alignment = center()
    return c

def style_input(ws, cell_ref, value, fmt=FMT_DOLLAR):
    c = ws[cell_ref]
    c.value = value
    c.font = font(color=C_BLUE)
    c.number_format = fmt
    c.alignment = right()
    return c

def style_formula(ws, cell_ref, formula, fmt=FMT_DOLLAR):
    c = ws[cell_ref]
    c.value = formula
    c.font = font(color=C_BLACK)
    c.number_format = fmt
    c.alignment = right()
    return c

def thin_bottom(ws, row, col_start, col_end):
    for col in range(col_start, col_end + 1):
        c = ws.cell(row=row, column=col)
        c.border = Border(top=c.border.top, left=c.border.left, right=c.border.right, bottom=Side(style="thin"))

def thick_bottom(ws, row, col_start, col_end):
    for col in range(col_start, col_end + 1):
        c = ws.cell(row=row, column=col)
        c.border = Border(top=c.border.top, left=c.border.left, right=c.border.right, bottom=Side(style="medium"))

def shade_row(ws, row, col_start, col_end, color=C_GREY):
    for col in range(col_start, col_end + 1):
        ws.cell(row=row, column=col).fill = fill(color)

# ────────────────────────────────────────────────────────────────────────────
# SHEET 1 — ASSUMPTIONS
# ────────────────────────────────────────────────────────────────────────────
def build_assumptions(wb):
    ws = wb.create_sheet("Assumptions")
    ws.sheet_view.showGridLines = False
    ws.column_dimensions["A"].width = 42
    ws.column_dimensions["B"].width = 18
    ws.column_dimensions["C"].width = 45
    ws.row_dimensions[1].height = 30

    ws.merge_cells("A1:C1")
    c = ws["A1"]
    c.value = "TESLA, INC. — DCF KEY ASSUMPTIONS"
    c.font = font(bold=True, size=14, color=C_WHITE)
    c.fill = fill(C_HEADER)
    c.alignment = center()

    def section(row, title):
        ws.merge_cells(f"A{row}:C{row}")
        c = ws[f"A{row}"]
        c.value = title
        c.font = font(bold=True, color=C_WHITE, size=10)
        c.fill = fill(C_SUBHEAD)
        c.alignment = left_al()

    def row_inp(row, label, value, fmt, note=""):
        ws[f"A{row}"].value = label
        ws[f"A{row}"].font = font()
        ws[f"A{row}"].alignment = Alignment(horizontal="left", vertical="center", indent=2)
        style_input(ws, f"B{row}", value, fmt)
        ws[f"C{row}"].value = note
        ws[f"C{row}"].font = font(italic=True, size=9, color="666666")
        if row % 2 == 0:
            shade_row(ws, row, 1, 3, C_GREY)

    # Headers row
    for col, h in zip(["A", "B", "C"], ["Assumption", "Value", "Notes / Source"]):
        c = ws[f"{col}3"]
        c.value = h
        c.font = font(bold=True, color=C_WHITE)
        c.fill = fill(C_HEADER)
        c.alignment = center()

    section(4, "REVENUE GROWTH — PROJECTION PERIOD (FY2025E–FY2029E)")
    row_inp(5,  "FY2025E Revenue Growth",  0.12, FMT_PCT, "Tesla 10-K FY2024; mgmt guidance ~10-15%")
    row_inp(6,  "FY2026E Revenue Growth",  0.18, FMT_PCT, "Cybertruck ramp + Next-gen platform intro")
    row_inp(7,  "FY2027E Revenue Growth",  0.22, FMT_PCT, "Full production scale; FSD/Network software mix")
    row_inp(8,  "FY2028E Revenue Growth",  0.20, FMT_PCT, "Energy Storage Megapack scaling")
    row_inp(9,  "FY2029E Revenue Growth",  0.18, FMT_PCT, "Commercial AI/Optimus initial contributions")

    section(11, "MARGIN ASSUMPTIONS")
    row_inp(12, "Gross Margin — FY2025E",  0.175, FMT_PCT, "Near-term automotive pricing headwinds")
    row_inp(13, "Gross Margin — FY2026E",  0.195, FMT_PCT, "Cost optimization + manufacturing efficiencies")
    row_inp(14, "Gross Margin — FY2027E",  0.210, FMT_PCT, "High-margin software revenue layer expansion")
    row_inp(15, "Gross Margin — FY2028E",  0.225, FMT_PCT, "Energy segment scaling economies")
    row_inp(16, "Gross Margin — FY2029E",  0.235, FMT_PCT, "Normalized long-term structural margin target")
    row_inp(17, "R&D as % of Revenue",     0.040, FMT_PCT, "Historical run-rate optimized for AI compute")
    row_inp(18, "SG&A as % of Revenue",    0.045, FMT_PCT, "Operational leverage scaling")

    section(20, "CAPEX, D&A & WORKING CAPITAL")
    row_inp(21, "D&A as % of Revenue",       0.055, FMT_PCT, "Reflects continuous Gigafactory expansion")
    row_inp(22, "CapEx as % of Revenue",     0.080, FMT_PCT, "AI Compute infrastructure + factory buildouts")
    row_inp(23, "Δ NWC as % of Rev Growth",  0.020, FMT_PCT, "Inventory and receivables working capital cycle")

    section(25, "TAX")
    row_inp(26, "Effective Tax Rate",  0.15, FMT_PCT, "Minimum global tax corporate optimization profile")

    section(28, "TERMINAL VALUE")
    row_inp(29, "Terminal Growth Rate (g)",  0.035, FMT_PCT,  "Long-run nominal GDP baseline plus secular tailwind")
    row_inp(30, "Exit EV/EBITDA Multiple",   25.0,  FMT_MULT, "Reflects unique technology and storage growth tier")

    section(32, "WACC INPUTS")
    row_inp(33, "Risk-Free Rate",          0.043,  FMT_PCT,  "10-Year US Treasury yield baseline")
    row_inp(34, "Equity Risk Premium",     0.055,  FMT_PCT,  "Damodaran Institutional ERP index")
    row_inp(35, "Beta (Levered)",          1.90,   '0.00',   "Systematic risk factor relative to global index")
    row_inp(36, "Pre-Tax Cost of Debt",    0.060,  FMT_PCT,  "Blended senior unsecured debt yield")
    row_inp(37, "Target Debt / (D+E)",     0.10,   FMT_PCT,  "Conservative target optimization level")

    section(39, "BALANCE SHEET BRIDGE (Latest, $mm)")
    row_inp(40, "Total Debt",          7400,  FMT_DOLLAR, "Recourse and non-recourse senior debt obligations")
    row_inp(41, "Cash & Equivalents", 36000,  FMT_DOLLAR, "Includes highly liquid cash assets and short-term paper")
    row_inp(42, "Minority Interest",      45,  FMT_DOLLAR, "Non-controlling interest segments")
    row_inp(43, "Shares Outstanding",  3195,  FMT_DOLLAR, "Fully diluted share capital structure base")
    row_inp(44, "FY2024A Revenue",    97690,  FMT_DOLLAR, "Historical reported revenue baseline")

    for r in [10, 19, 24, 27, 31, 38]:
        ws.row_dimensions[r].height = 6

    thick_bottom(ws, 44, 1, 3)
    ws.freeze_panes = "A4"
    return ws

# ────────────────────────────────────────────────────────────────────────────
# SHEET 2 — WACC MATRIX
# ────────────────────────────────────────────────────────────────────────────
def build_wacc(wb):
    ws = wb.create_sheet("WACC")
    ws.sheet_view.showGridLines = False
    ws.column_dimensions["A"].width = 38
    ws.column_dimensions["B"].width = 18
    ws.column_dimensions["C"].width = 40
    ws.row_dimensions[1].height = 28

    ws.merge_cells("A1:C1")
    c = ws["A1"]
    c.value = "WEIGHTED AVERAGE COST OF CAPITAL (WACC)"
    c.font = font(bold=True, size=13, color=C_WHITE)
    c.fill = fill(C_HEADER)
    c.alignment = center()

    def row(r, label, formula_str, fmt, note=""):
        ws.cell(row=r, column=1, value=label).font = font(bold=False)
        ws.cell(row=r, column=1).alignment = Alignment(horizontal="left", vertical="center", indent=2)
        style_formula(ws, f"B{r}", formula_str, fmt)
        if note:
            ws.cell(row=r, column=3, value=note).font = font(italic=True, size=9, color="666666")
        if r % 2 == 0:
            shade_row(ws, r, 1, 3, C_GREY)

    for col, h in zip([1,2,3], ["Item", "Value", "Note / Component Logic"]):
        c = ws.cell(row=3, column=col, value=h)
        c.font = font(bold=True, color=C_WHITE)
        c.fill = fill(C_HEADER)
        c.alignment = center()

    row(4,  "Risk-Free Rate (Rf)",           "=Assumptions!B33", FMT_PCT,  "Linked to Assumptions")
    row(5,  "Equity Risk Premium (ERP)",      "=Assumptions!B34", FMT_PCT,  "Linked to Assumptions")
    row(6,  "Levered Beta",                   "=Assumptions!B35", "0.00",   "Linked to Assumptions")
    row(7,  "Cost of Equity (CAPM)",          "=B4+B6*B5",        FMT_PCT,  "Ke = Rf + (Beta × ERP)")
    row(8,  "Pre-Tax Cost of Debt (Kd)",      "=Assumptions!B36", FMT_PCT,  "Linked to Assumptions")
    row(9,  "Tax Rate",                       "=Assumptions!B26", FMT_PCT,  "Linked to Assumptions")
    row(10, "After-Tax Cost of Debt",         "=B8*(1-B9)",       FMT_PCT,  "Kd × (1 – t)")
    row(11, "Debt Weight [D/(D+E)]",          "=Assumptions!B37", FMT_PCT,  "Capital structure target constraint")
    row(12, "Equity Weight [E/(D+E)]",        "=1-B11",           FMT_PCT,  "Complement weight (1 - Wd)")

    ws.merge_cells("A13:C13")
    ws["A13"].fill = fill(C_LIGHT)

    row(14, "Calculated WACC",                 "=B7*B12+B10*B11", FMT_PCT,  "Blended corporate hurdle discount rate")
    ws.cell(row=14, column=1).font = font(bold=True)
    ws.cell(row=14, column=2).font = font(bold=True, color=C_SUBHEAD)
    ws.cell(row=14, column=2).fill = fill(C_YELLOW)
    for col in [1, 3]:
        ws.cell(row=14, column=col).fill = fill(C_LIGHT)

    thick_bottom(ws, 14, 1, 3)
    ws.freeze_panes = "A4"
    return ws

# ────────────────────────────────────────────────────────────────────────────
# SHEET 3 — DCF MODEL
# ────────────────────────────────────────────────────────────────────────────
def build_dcf(wb):
    ws = wb.create_sheet("DCF Model")
    ws.sheet_view.showGridLines = False

    ws.column_dimensions["A"].width = 40
    for col in ["B", "C", "D", "E", "F", "G"]:
        ws.column_dimensions[col].width = 15
    ws.column_dimensions["H"].width = 16   # Terminal Value Matrix col
    ws.column_dimensions["I"].width = 20

    years     = ["FY2024A", "FY2025E", "FY2026E", "FY2027E", "FY2028E", "FY2029E"]
    year_cols = ["B", "C", "D", "E", "F", "G"]
    TV_COL    = "H"

    ws.merge_cells("A1:I1")
    c = ws["A1"]
    c.value = "TESLA, INC. (TSLA) — UNLEVERED DCF VALUATION MODEL"
    c.font = font(bold=True, size=13, color=C_WHITE)
    c.fill = fill(C_HEADER)
    c.alignment = center()
    ws.row_dimensions[1].height = 28

    ws.merge_cells("A2:I2")
    c = ws["A2"]
    c.value = "Figures in $mm unless stated | Fiscal Year Ends Dec 31 | Valuation Core Matrix"
    c.font = font(italic=True, size=9, color="444444")
    c.fill = fill(C_LIGHT)
    c.alignment = center()

    ROW_HDR = 4
    ws.row_dimensions[ROW_HDR].height = 24
    style_header_cell(ws, f"A{ROW_HDR}", "Valuation Bridge Elements", dark=True)
    for yr, col in zip(years, year_cols):
        style_header_cell(ws, f"{col}{ROW_HDR}", yr, dark=True)
    style_header_cell(ws, f"{TV_COL}{ROW_HDR}", "Terminal Phase", dark=True)

    def label(row, txt, bold=False, indent=1, italic=False):
        c = ws.cell(row=row, column=1, value=txt)
        c.font = font(bold=bold, italic=italic)
        c.alignment = Alignment(horizontal="left", vertical="center", indent=indent)

    COL = {y: i+2 for i, y in enumerate(years)}  # B=2, C=3...
    TV_C = 8

    # Dynamic structure anchors
    R_REV   = 6;  R_RGROW = 7;  R_GP    = 8;  R_GPM   = 9;  R_RD    = 10
    R_SGA   = 11; R_EBIT  = 12; R_EBITM = 13; R_DA    = 14; R_EBITDA= 15
    R_EBITDM= 16; R_TAX   = 17; R_NOPAT = 18

    ws.merge_cells("A5:I5")
    ws["A5"] = "INCOME STATEMENT & COMPONENT EBITDA EXTRACTION"; ws["A5"].font = font(bold=True, color=C_WHITE); ws["A5"].fill = fill(C_SUBHEAD)

    label(R_REV, "Revenue", bold=True)
    style_formula(ws, f"B{R_REV}", "=Assumptions!B44", FMT_DOLLAR) # link to historical base

    for i, yr in enumerate(years[1:], 5):
        col_l = year_cols[i-4]
        prev_l = year_cols[i-5]
        style_formula(ws, f"{col_l}{R_REV}", f"={prev_l}{R_REV}*(1+Assumptions!B{i})", FMT_DOLLAR)

    label(R_RGROW, "  YoY Growth Rate", italic=True)
    for i, yr in enumerate(years[1:], 5):
        col_l = year_cols[i-4]
        prev_l = year_cols[i-5]
        style_formula(ws, f"{col_l}{R_RGROW}", f"={col_l}{R_REV}/{prev_l}{R_REV}-1", FMT_PCT)

    label(R_GP, "Gross Profit", bold=True)
    style_formula(ws, f"B{R_GP}", "=17107", FMT_DOLLAR) # actual historic margin conversion
    for i, yr in enumerate(years[1:], 12):
        col_l = year_cols[i-11]
        style_formula(ws, f"{col_l}{R_GP}", f"={col_l}{R_REV}*Assumptions!B{i}", FMT_DOLLAR)

    label(R_GPM, "  Gross Margin %", italic=True)
    for col_l in year_cols:
        style_formula(ws, f"{col_l}{R_GPM}", f"={col_l}{R_GP}/{col_l}{R_REV}", FMT_PCT)

    label(R_RD, "R&D Expense")
    style_formula(ws, f"B{R_RD}", "=3976", FMT_DOLLAR)
    for col_l in year_cols[1:]:
        style_formula(ws, f"{col_l}{R_RD}", f"={col_l}{R_REV}*Assumptions!B17", FMT_DOLLAR)

    label(R_SGA, "SG&A Expense")
    style_formula(ws, f"B{R_SGA}", "=4350", FMT_DOLLAR)
    for col_l in year_cols[1:]:
        style_formula(ws, f"{col_l}{R_SGA}", f"={col_l}{R_REV}*Assumptions!B18", FMT_DOLLAR)

    label(R_EBIT, "EBIT (Operating Income)", bold=True)
    for col_l in year_cols:
        style_formula(ws, f"{col_l}{R_EBIT}", f"={col_l}{R_GP}-{col_l}{R_RD}-{col_l}{R_SGA}", FMT_DOLLAR)

    label(R_EBITM, "  Operating Margin %", italic=True)
    for col_l in year_cols:
        style_formula(ws, f"{col_l}{R_EBITM}", f"={col_l}{R_EBIT}/{col_l}{R_REV}", FMT_PCT)

    label(R_DA, "Depreciation & Amortization")
    style_formula(ws, f"B{R_DA}", "=5310", FMT_DOLLAR)
    for col_l in year_cols[1:]:
        style_formula(ws, f"{col_l}{R_DA}", f"={col_l}{R_REV}*Assumptions!B21", FMT_DOLLAR)

    label(R_EBITDA, "EBITDA", bold=True)
    for col_l in year_cols:
        style_formula(ws, f"{col_l}{R_EBITDA}", f"={col_l}{R_EBIT}+{col_l}{R_DA}", FMT_DOLLAR)

    label(R_TAX, "Taxes on EBIT")
    for col_l in year_cols:
        style_formula(ws, f"{col_l}{R_TAX}", f"={col_l}{R_EBIT}*Assumptions!B26", FMT_DOLLAR)

    label(R_NOPAT, "NOPAT", bold=True)
    for col_l in year_cols:
        style_formula(ws, f"{col_l}{R_NOPAT}", f"={col_l}{R_EBIT}-{col_l}{R_TAX}", FMT_DOLLAR)

    # Shading and highlights
    for r in [R_RGROW, R_GPM, R_EBITM]:
        shade_row(ws, r, 1, 8, C_GREY)
    for r in [R_EBIT, R_EBITDA, R_NOPAT]:
        shade_row(ws, r, 1, 8, C_LIGHT)

    # ── FREE CASH FLOW EXTENSION BLOCK ──────────────────────────────────────────
    R_FCF_HDR = 20; R_NOPAT2 = 21; R_DA2 = 22; R_CAPEX = 23; R_NWC = 24
    R_UFCF = 25; R_MID = 26; R_PUFCF = 27

    ws.merge_cells(f"A{R_FCF_HDR}:I{R_FCF_HDR}")
    ws[f"A{R_FCF_HDR}"] = "UNLEVERED FREE CASH FLOW PROFILE (UFCF)"; ws[f"A{R_FCF_HDR}"].font = font(bold=True, color=C_WHITE); ws[f"A{R_FCF_HDR}"].fill = fill(C_SUBHEAD)

    label(R_NOPAT2, "NOPAT", bold=True)
    for col_l in year_cols:
        style_formula(ws, f"{col_l}{R_NOPAT2}", f"={col_l}{R_NOPAT}", FMT_DOLLAR)

    label(R_DA2, "(+) Depreciation & Amortization")
    for col_l in year_cols:
        style_formula(ws, f"{col_l}{R_DA2}", f"={col_l}{R_DA}", FMT_DOLLAR)

    label(R_CAPEX, "(–) Capital Expenditures")
    style_formula(ws, f"B{R_CAPEX}", "=-7853", FMT_DOLLAR)
    for col_l in year_cols[1:]:
        style_formula(ws, f"{col_l}{R_CAPEX}", f"=-{col_l}{R_REV}*Assumptions!B22", FMT_DOLLAR)

    label(R_NWC, "(–) Increase in Net Working Capital")
    style_formula(ws, f"B{R_NWC}", "=-412", FMT_DOLLAR)
    for i, col_l in enumerate(year_cols[1:], 1):
        prev_l = year_cols[i-1]
        style_formula(ws, f"{col_l}{R_NWC}", f"=-({col_l}{R_REV}-{prev_l}{R_REV})*Assumptions!B23", FMT_DOLLAR)

    label(R_UFCF, "Unlevered Free Cash Flow (UFCF)", bold=True)
    for col_l in year_cols:
        style_formula(ws, f"{col_l}{R_UFCF}", f"={col_l}{R_NOPAT2}+{col_l}{R_DA2}+{col_l}{R_CAPEX}+{col_l}{R_NWC}", FMT_DOLLAR)
    shade_row(ws, R_UFCF, 1, 8, C_LIGHT)

    label(R_MID, "Mid-Year Discount Interval Period", italic=True)
    for i, col_l in enumerate(year_cols[1:], 1):
        style_formula(ws, f"{col_l}{R_MID}", f"={i}-0.5", '0.0')
    shade_row(ws, R_MID, 1, 8, C_GREY)

    label(R_PUFCF, "Present Value of UFCF", bold=True)
    for col_l in year_cols[1:]:
        style_formula(ws, f"{col_l}{R_PUFCF}", f"={col_l}{R_UFCF}/(1+WACC!B14)^{col_l}{R_MID}", FMT_DOLLAR)
    shade_row(ws, R_PUFCF, 1, 8, C_LIGHT)

    # ── TERMINAL VALUE EVALUATIONS ──────────────────────────────────────────────
    R_TV_HDR = 29; R_GGM = 30; R_MULT = 31; R_AVG = 32; R_PV_TV = 33
    ws.merge_cells(f"A{R_TV_HDR}:I{R_TV_HDR}")
    ws[f"A{R_TV_HDR}"] = "TERMINAL MATRIX INTEGRATION"; ws[f"A{R_TV_HDR}"].font = font(bold=True, color=C_WHITE); ws[f"A{R_TV_HDR}"].fill = fill(C_SUBHEAD)

    label(R_GGM, "Gordon Growth Model Terminal Base")
    style_formula(ws, f"H{R_GGM}", f"=G{R_UFCF}*(1+Assumptions!B29)/(WACC!B14-Assumptions!B29)", FMT_DOLLAR)

    label(R_MULT, "EV/EBITDA Exit Multiple Formulation")
    style_formula(ws, f"H{R_MULT}", f"=G{R_EBITDA}*Assumptions!B30", FMT_DOLLAR)

    label(R_AVG, "Blended Target Terminal Value [USED]", bold=True)
    style_formula(ws, f"H{R_AVG}", f"=AVERAGE(H{R_GGM},H{R_MULT})", FMT_DOLLAR)
    ws.cell(row=R_AVG, column=8).fill = fill(C_LIGHT)

    label(R_PV_TV, "PV of Terminal Perpetuity Target")
    style_formula(ws, f"H{R_PV_TV}", f"=H{R_AVG}/(1+WACC!B14)^5", FMT_DOLLAR)

    # ── VALUATION CONSOLIDATION AND BRIDGE ──────────────────────────────────────
    R_VAL_HDR = 35; R_SUM_FCF = 36; R_ADD_TV = 37; R_EV = 38; R_DEBT = 39
    R_CASH = 40; R_MI = 41; R_EQ = 42; R_SO = 43; R_PRICE = 44

    ws.merge_cells(f"A{R_VAL_HDR}:I{R_VAL_HDR}")
    ws[f"A{R_VAL_HDR}"] = "EQUITY VALUATION BRIDGE STRUCTURE"; ws[f"A{R_VAL_HDR}"].font = font(bold=True, color=C_WHITE); ws[f"A{R_VAL_HDR}"].fill = fill(C_SUBHEAD)

    label(R_SUM_FCF, "(+) Sum of PV of UFCFs (FY2025E–FY2029E)")
    style_formula(ws, f"B{R_SUM_FCF}", f"=SUM(C{R_PUFCF}:G{R_PUFCF})", FMT_DOLLAR)

    label(R_ADD_TV, "(+) PV of Terminal Value Contribution")
    style_formula(ws, f"B{R_ADD_TV}", f"=H{R_PV_TV}", FMT_DOLLAR)

    label(R_EV, "Enterprise Value (EV)", bold=True)
    style_formula(ws, f"B{R_EV}", f"=B{R_SUM_FCF}+B{R_ADD_TV}", FMT_DOLLAR)
    shade_row(ws, R_EV, 1, 2, C_LIGHT)

    label(R_DEBT, "(–) Total Debt Outstanding Balance")
    style_formula(ws, f"B{R_DEBT}", "=-Assumptions!B40", FMT_DOLLAR)

    label(R_CASH, "(+) Corporate Cash & Short Term Assets")
    style_formula(ws, f"B{R_CASH}", "=Assumptions!B41", FMT_DOLLAR)

    label(R_MI, "(–) Non-Controlling Minority Interests")
    style_formula(ws, f"B{R_MI}" if 'B_MI' not in locals() else f"B{R_MI}", "=-Assumptions!B42", FMT_DOLLAR)

    label(R_EQ, "Implied Corporate Equity Value", bold=True)
    style_formula(ws, f"B{R_EQ}", f"=B{R_EV}+B{R_DEBT}+B{R_CASH}+B{R_MI}", FMT_DOLLAR)
    shade_row(ws, R_EQ, 1, 2, C_LIGHT)

    label(R_SO, "Diluted Shares Outstanding Scale (mm)")
    style_formula(ws, f"B{R_SO}", "=Assumptions!B43", FMT_DOLLAR)

    label(R_PRICE, "Implied Target Share Price (USD)", bold=True)
    style_formula(ws, f"B{R_PRICE}", f"=B{R_EQ}/B{R_SO}", '$#,##0.00')
    ws.cell(row=R_PRICE, column=1).font = font(bold=True, size=11)
    ws.cell(row=R_PRICE, column=2).font = font(bold=True, size=12, color=C_SUBHEAD)
    ws.cell(row=R_PRICE, column=2).fill = fill(C_YELLOW)

    for r in [R_EV, R_EQ, R_PRICE]:
        thick_bottom(ws, r, 1, 2)

    ws.freeze_panes = "B5"
    return ws

# ────────────────────────────────────────────────────────────────────────────
# SHEET 4 — SENSITIVITY MATRIX SYSTEM
# ────────────────────────────────────────────────────────────────────────────
def build_sensitivity(wb):
    ws = wb.create_sheet("Sensitivity")
    ws.sheet_view.showGridLines = False
    ws.column_dimensions["A"].width = 24

    ws.merge_cells("A1:G1")
    c = ws["A1"]
    c.value = "SENSITIVITY DATA MATRIX — TSLA VALUE PER SHARE"
    c.font = font(bold=True, size=12, color=C_WHITE)
    c.fill = fill(C_HEADER)
    c.alignment = center()
    ws.row_dimensions[1].height = 26

    ws["A3"] = "WACC  ↓  /  Terminal Growth Rate  →"
    ws["A3"].font = font(bold=True, italic=True)

    waccs = [0.09, 0.10, 0.11, 0.12, 0.13]
    tgrs  = [0.025, 0.030, 0.035, 0.040, 0.045]

    for j, tgr in enumerate(tgrs):
        col = j + 2
        c = ws.cell(row=4, column=col, value=tgr)
        c.font = font(bold=True, color=C_WHITE)
        c.fill = fill(C_SUBHEAD)
        c.number_format = FMT_PCT
        c.alignment = center()
        ws.column_dimensions[get_column_letter(col)].width = 14

    for i, wacc in enumerate(waccs):
        row_i = 5 + i
        c = ws.cell(row=row_i, column=1, value=wacc)
        c.font = font(bold=True, color=C_WHITE)
        c.fill = fill(C_SUBHEAD)
        c.number_format = FMT_PCT
        c.alignment = center()

        for j, tgr in enumerate(tgrs):
            col = j + 2
            cell = ws.cell(row=row_i, column=col)
            # Re-engineered mathematically sound self-contained cell string
            cell.value = (
                f"=('DCF Model'!B36"
                f"+(('DCF Model'!G25*(1+{tgr})/({wacc}-{tgr}))/(1+{wacc})^5)"
                f"+Assumptions!B41-Assumptions!B40-Assumptions!B42)"
                f"/Assumptions!B43"
            )
            cell.number_format = '$#,##0.00'
            cell.font = font()
            cell.alignment = center()
            if abs(wacc - 0.11) < 0.001 and abs(tgr - 0.035) < 0.001:
                cell.fill = fill(C_YELLOW)

    ws["A4"] = "WACC \\ TGR"; ws["A4"].font = font(bold=True, color=C_WHITE); ws["A4"].fill = fill(C_HEADER); ws["A4"].alignment = center()

    # Table 2: Multiple Structure Variation Matrix
    ws["A12"] = "WACC  ↓  /  EV/EBITDA Exit Multiple  →"
    ws["A12"].font = font(bold=True, italic=True)

    multiples = [18, 20, 22, 25, 28, 30]
    for j, m in enumerate(multiples):
        col = j + 2
        c = ws.cell(row=13, column=col, value=m)
        c.font = font(bold=True, color=C_WHITE)
        c.fill = fill(C_SUBHEAD)
        c.number_format = '0"x"'
        c.alignment = center()
        ws.column_dimensions[get_column_letter(col)].width = 14

    ws["A13"] = "WACC \\ EV/EBITDA"; ws["A13"].font = font(bold=True, color=C_WHITE); ws["A13"].fill = fill(C_HEADER); ws["A13"].alignment = center()

    for i, wacc in enumerate(waccs):
        row_i = 14 + i
        c = ws.cell(row=row_i, column=1, value=wacc)
        c.font = font(bold=True, color=C_WHITE)
        c.fill = fill(C_SUBHEAD)
        c.number_format = FMT_PCT
        c.alignment = center()

        for j, m in enumerate(multiples):
            col = j + 2
            cell = ws.cell(row=row_i, column=col)
            cell.value = (
                f"=('DCF Model'!B36"
                f"+(('DCF Model'!G15*{m})/(1+{wacc})^5)"
                f"+Assumptions!B41-Assumptions!B40-Assumptions!B42)"
                f"/Assumptions!B43"
            )
            cell.number_format = '$#,##0.00'
            cell.font = font()
            cell.alignment = center()
            if abs(wacc - 0.11) < 0.001 and m == 25:
                cell.fill = fill(C_YELLOW)

    ws.freeze_panes = "B5"
    return ws

def build_cover(wb):
    ws = wb.create_sheet("Cover", 0)
    ws.sheet_view.showGridLines = False
    for col in ["A","B","C","D"]:
        ws.column_dimensions[col].width = 24

    ws.merge_cells("A1:D1")
    c = ws["A1"]
    c.value = "TESLA, INC. (TSLA)"
    c.font = font(bold=True, size=22, color=C_WHITE)
    c.fill = fill(C_HEADER)
    c.alignment = center()
    ws.row_dimensions[1].height = 50

    ws.merge_cells("A2:D2")
    c = ws["A2"]
    c.value = "Institutional Discounted Cash Flow (DCF) Architecture Model"
    c.font = font(bold=True, size=12, color=C_WHITE)
    c.fill = fill(C_SUBHEAD)
    c.alignment = center()
    ws.row_dimensions[2].height = 30

# ────────────────────────────────────────────────────────────────────────────
# RUNTIME EXECUTIVE PIPELINE
# ────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    wb = Workbook()

    # Run creation sequence
    build_cover(wb)
    build_assumptions(wb)
    build_wacc(wb)
    build_dcf(wb)
    build_sensitivity(wb)

    # Remove default workbook initialization placeholder artifact
    if "Sheet" in wb.sheetnames:
        wb.remove(wb["Sheet"])

    output_filename = "TSLA_Institutional_DCF_Model.xlsx"
    wb.save(output_filename)
    print(f"Success. Model output generation initialized. Saved as: '{output_filename}'")

Success. Model output generation initialized. Saved as: 'TSLA_Institutional_DCF_Model.xlsx'
